# Bimodal λ test: what each rule does to the estimated θ

Results of `bimodal_lamtest.py` (launched with `launch_bimodal.sh`). Each **run** is one dataset of
n samples from the paper's bimodal law at one β (= |θ| in the paper's figure), pushed through MGD;
the per-step multipliers θ_t are then smoothed in time with every λ of the grid, and each
**rule** picks one λ. Because the true law is known exactly, every estimate can be compared with
the truth.

What is compared with what:

* **θ\*** – the true multipliers of the law the data were drawn from.
* **max-likelihood fit (MLE)** – the max-ent fit of the data's own moments. No method can do better
  from these n samples, so it is the **floor**.
* **raw MGD θ_t** – no smoothing (λ = 0).
* **the rules** – even/odd CV (the current rule), block CV, Guth's weights at λ = 1, and the
  *oracle* (the best λ of the grid, found with θ\*; not usable in practice, it is the yardstick).

Error measure: the paper's D_KL(p_θ\* ‖ p_θ̂) at the end of the transport, computed exactly on a grid.

Sections: 1 summary table · 2 error vs β (paper-style) · 3 θ(t) along the transport · 4 final θ̂
against θ\* · 5 the estimated densities and the weight of each mode · 6 error vs λ · 7 chosen vs best λ ·
8 is the noise of θ_t correlated in time?

In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / 'codes').is_dir():          # works from the repo root or from lam_selection_criterion/
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / 'codes'), str(ROOT / 'data'), str(ROOT / 'lam_selection_criterion')]
import bimodal_lamtest as bl

# results pulled from Jean Zay (or set BIMODAL_LAMTEST_OUT)
OUT = Path(os.environ.get('BIMODAL_LAMTEST_OUT', ROOT / 'lam_selection_criterion' / 'results' / 'bimodal'))
cfg = json.loads((OUT / 'args.json').read_text())
runs = {}
for bdir in sorted(OUT.glob('beta_*')):
    rs = [torch.load(f, map_location='cpu', weights_only=False) for f in sorted(bdir.glob('run_*.pt'))]
    if rs:
        runs[rs[0]['beta']] = rs
betas = sorted(runs)
BLOCKS = cfg['blocks']
print(f"{OUT}\nn = {cfg['n']}, sigma = {cfg['sigma']}, nt = {cfg['nt']}, ridge = {cfg['regularization']}, "
      f"lams = {cfg['lams']}, blocks = {BLOCKS}")
for b in betas:
    print(f"beta {b:.2f}: {len(runs[b])} runs, {runs[b][0]['n_nodes']} nodes, "
          f"mean run time {np.mean([r['t_total_s'] for r in runs[b]]) / 60:.1f} min")

In [ ]:
# style: reference palette slots 1-4 in fixed order + one line style / marker per rule (never colour alone)
INK, INK2, MUTED, GRIDC, SURF = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#fcfcfb'
BMAX = max(BLOCKS)
SHOW = {   # rule -> (label, colour, line style, marker)
    'lam0':                 ('raw MGD θ_t (λ = 0)',            MUTED,     '-',  'o'),
    'uniform_cv_eo':        ('even/odd CV (current rule)',     '#2a78d6', '-',  's'),
    f'uniform_cv_block{BMAX}': (f'block CV, B = {BMAX}',        '#eb6834', '--', 'D'),
    'guth_lam1':            ('Guth weights, λ = 1',            '#1baf7a', ':',  '^'),
    'uniform_oracle_end':   ('best λ of the grid (oracle)',    '#eda100', '-.', 'v'),
}
ALL_RULES = bl.rule_order(BLOCKS)
COMP = ['θ₁  (x)', 'θ₂  (x²/2)', 'θ₃  (x³/3)', 'θ₄  (x⁴/4)']
plt.rcParams.update({'figure.facecolor': SURF, 'axes.facecolor': SURF, 'axes.edgecolor': '#c3c2b7',
                     'axes.labelcolor': INK2, 'xtick.color': MUTED, 'ytick.color': MUTED, 'text.color': INK,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': GRIDC, 'grid.linewidth': 0.6, 'font.size': 11, 'legend.frameon': False,
                     'lines.linewidth': 2.0, 'lines.markersize': 8})


def key_of(res, rule):
    d = res['rules'].get(rule, {})
    if 'end_true' not in d:
        return None
    return 'theta_t' if rule == 'lam0' else f"{d['sched']}|{d['lam']:g}"


def theta_end(res, rule):
    k = key_of(res, rule)
    return None if k is None else res['theta_end'][k].double()


def theta_curve(res, rule):
    k = key_of(res, rule)
    return None if k is None else (res['plot']['theta_t'] if k == 'theta_t' else res['plot']['Theta'][k])


def legend_above(fig, handles, labels, ncol=3, title=None):
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=ncol, fontsize=10,
               title=title, title_fontsize=12)

## 1. Summary table

One row per (β, rule). **λ chosen**: which λ the rule picked, with how many runs picked it.
**KL mean / median**: the error D_KL(p_θ\* ‖ p_θ̂) at the end of the transport, over runs.
**÷ floor**: mean error divided by the mean error of the max-likelihood fit. 1 is the best possible.
**not normalisable**: runs where the estimate has θ₄ ≥ 0, so p_θ̂ is not a density and the KL is
infinite. **mid-run KL**: the same kind of error, averaged over 0.05 ≤ t < 0.95 (against the max-ent
fit of the interpolant's moments at each t).

In [ ]:
rows = []
for b in betas:
    floor = np.mean([r['kl_floor'] for r in runs[b]])
    for rule in ALL_RULES:
        ds = [r['rules'].get(rule, {}) for r in runs[b]]
        ds = [d for d in ds if 'end_true' in d]
        if not ds:
            continue
        e = np.array([d['end_true'] for d in ds]); f = e[np.isfinite(e)]
        lams = pd.Series([d['lam'] for d in ds]).value_counts().sort_index()
        rows.append({'beta': b, 'rule': rule,
                     'λ chosen (runs)': ', '.join(f'{l:.0e}×{c}' for l, c in lams.items()),
                     'KL mean': f.mean() if len(f) else np.nan, 'KL median': np.median(e),
                     '÷ floor': f.mean() / floor if len(f) else np.nan,
                     'not normalisable': int((~np.isfinite(e)).sum()),
                     'mid-run KL': np.median([d['[0.05,0.95)'] for d in ds])})
    rows.append({'beta': b, 'rule': 'floor: max-likelihood fit', 'KL mean': floor, '÷ floor': 1.0,
                 'KL median': np.median([r['kl_floor'] for r in runs[b]])})
summary = pd.DataFrame(rows).set_index(['beta', 'rule'])
pd.set_option('display.float_format', lambda v: f'{v:.3g}')
summary

## 2. Error vs β, as in the paper's Figure 1

Left: error to the **true** law, D_KL(p_θ\* ‖ p_θ̂). Markers are the median over runs, bars the
25-75 % range. The dashed line is the floor (max-likelihood fit), the dotted line r/2n (the
Cramér-Rao value the floor should approach for large n).
Right: error to the **mean of the runs**, D_KL(p_θ̄ ‖ p_θ̂), the quantity of the paper's figure.
It only measures how much the estimates scatter between datasets, not whether they are right.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
xb = np.array(betas)
for rule, (lab, col, ls, mk) in SHOW.items():
    med, lo, hi, med2 = [], [], [], []
    for b in betas:
        e = np.array([r['rules'].get(rule, {}).get('end_true', np.nan) for r in runs[b]])
        med.append(np.median(e)); q = np.nanpercentile(np.where(np.isfinite(e), e, np.nan), [25, 75]); lo.append(q[0]); hi.append(q[1])
        th = [theta_end(r, rule) for r in runs[b]]
        th = torch.stack([x for x in th if x is not None])
        e2 = bl.kl_exact(th.mean(0), th).numpy()
        med2.append(np.median(e2))
    med, lo, hi = map(np.array, (med, lo, hi))
    axs[0].errorbar(xb, med, yerr=[med - lo, hi - med], color=col, ls=ls, marker=mk, mfc=SURF, mew=2,
                    capsize=3, elinewidth=1, label=lab)
    axs[1].plot(xb, med2, color=col, ls=ls, marker=mk, mfc=SURF, mew=2, label=lab)
floor = [np.mean([r['kl_floor'] for r in runs[b]]) for b in betas]
axs[0].plot(xb, floor, color=INK, ls='--', lw=1.5, label='floor: max-likelihood fit')
for ax in axs:
    ax.axhline(runs[betas[0]][0]['cr_bound'], color=MUTED, ls=':', lw=1.5, label='r / 2n')
    ax.set_yscale('log'); ax.set_xlabel('β  (= |θ| in the paper)')
axs[0].set_ylabel('D_KL(p_θ* ‖ p_θ̂) at t = 1'); axs[1].set_ylabel('D_KL(p_θ̄ ‖ p_θ̂)  (spread between runs)')
h, l = axs[0].get_legend_handles_labels()
legend_above(fig, h, l, ncol=4)
plt.show()

## 3. θ(t) along the transport, one run

Each column is one component of θ. **Top row**: the whole transport in t. **Bottom row**: the end of
the transport, with 1 - t on a log axis (t = 1 is on the right), which is where the per-step
estimate gets very noisy.

* grey: the raw per-step MGD θ_t. Near t = 1 it leaves the frame; the axes follow the smooth curves.
* black dashed: the max-ent fit of the interpolant's moments at each t (the reference). It is blank
  where that fit does not exist on the whole line (θ₄ ≥ 0, see the script's docstring).
* black dotted horizontal: θ\*, the true value at t = 1.
* coloured: the smoothed Θ(t) at the λ each rule picked for this run.

Change `BETA_SHOW` / `RUN_SHOW` to look at another run.

In [ ]:
BETA_SHOW = betas[len(betas) // 2]
RUN_SHOW = 0
res = runs[BETA_SHOW][RUN_SHOW]
P = res['plot']
t = P['t'].numpy(); x_end = np.maximum(1 - t, 1e-7)
ref = P['theta_me'].numpy().copy(); ref[~P['valid'].numpy()] = np.nan
fig, axs = plt.subplots(2, 4, figsize=(17, 7.5), constrained_layout=True)
handles = {}
for a in range(4):
    for row, (ax, xx) in enumerate(((axs[0, a], t), (axs[1, a], x_end))):
        ax.plot(xx, P['theta_t'][:, a], color=MUTED, lw=0.6, alpha=0.7)
        h_ref, = ax.plot(xx, ref[:, a], color=INK, ls='--', lw=1.4)
        ys = [ref[:, a][np.isfinite(ref[:, a])], [float(res['theta_star'][a])]]
        for rule, (lab, col, ls, mk) in SHOW.items():
            if rule == 'lam0':
                continue
            c = theta_curve(res, rule)
            if c is None:
                continue
            lam = res['rules'][rule]['lam']
            handles[rule], = ax.plot(xx, c[:, a], color=col, ls=ls, lw=1.8)
            ys.append(c[:, a].numpy())
        h_star = ax.axhline(float(res['theta_star'][a]), color=INK, ls=':', lw=1.2)
        lo, hi = np.min(np.concatenate(ys)), np.max(np.concatenate(ys)); pad = 0.25 * (hi - lo + 1e-12)
        ax.set_ylim(lo - pad, hi + pad)
        if row == 1:
            ax.set_xscale('log'); ax.invert_xaxis(); ax.set_xlabel('1 - t')
        else:
            ax.set_xlabel('t'); ax.set_title(COMP[a], color=INK)
h_raw = plt.Line2D([], [], color=MUTED, lw=1)
labs = ['raw MGD θ_t', 'reference (max-ent fit of m_t)', 'θ* (true, t = 1)'] + \
       [f"{SHOW[k][0]}: λ = {res['rules'][k]['lam']:.0e}" for k in handles]
legend_above(fig, [h_raw, h_ref, h_star] + list(handles.values()), labs, ncol=4,
             title=f'β = {BETA_SHOW:.2f}, run {RUN_SHOW}')
plt.show()

## 4. Final θ̂ against θ\*, all runs

For each component, the error θ̂ - θ\* at t = 1, one dot per run, for every rule and for the
max-likelihood fit (the floor). The black bar is the mean over runs: if it sits away from 0, the
rule is **biased**; the spread of the dots is its **variance**. The y axis is linear near 0 and
logarithmic further out, so the raw MGD θ_t (usually far off) stays on the plot.

In [ ]:
cols = list(SHOW) + ['mle']
labels = [SHOW[c][0] for c in SHOW] + ['max-likelihood fit (floor)']
fig, axs = plt.subplots(len(betas), 4, figsize=(17, 3.6 * len(betas)), constrained_layout=True, squeeze=False)
rng = np.random.default_rng(0)
for i, b in enumerate(betas):
    for a in range(4):
        ax = axs[i, a]
        mle_err = np.array([float(r['theta_mle'][a] - r['theta_star'][a]) for r in runs[b]])
        lin = 3 * max(np.abs(mle_err).max(), 1e-12)
        for j, c in enumerate(cols):
            if c == 'mle':
                e, col, mk = mle_err, INK2, 'o'
            else:
                e = np.array([np.nan if theta_end(r, c) is None else float(theta_end(r, c)[a] - r['theta_star'][a])
                              for r in runs[b]])
                col, mk = SHOW[c][1], SHOW[c][3]
            ax.scatter(j + rng.uniform(-0.15, 0.15, len(e)), e, color=col, marker=mk, s=28, alpha=0.8,
                       edgecolors=SURF, linewidths=0.8, zorder=3)
            ax.hlines(np.nanmean(e), j - 0.3, j + 0.3, color=INK, lw=2, zorder=4)
        ax.axhline(0, color=INK2, lw=0.8)
        ax.set_yscale('symlog', linthresh=lin)
        ax.set_xticks(range(len(cols)))
        ax.set_xticklabels(['raw', 'eo CV', f'block {BMAX}', 'Guth λ=1', 'oracle', 'MLE'] if i == len(betas) - 1 else [],
                           rotation=30)
        if i == 0:
            ax.set_title(COMP[a], color=INK)
        if a == 0:
            ax.set_ylabel(f'β = {b:.2f}\nθ̂ - θ*')
plt.show()

## 5. The estimated densities, and the weight of each mode

Left plots: log p_θ̂(x) at t = 1 for one run per β, against the true log p\* (thick black) and the
max-likelihood fit (dashed). The draft's point is that per-step score matching cannot get the
**relative weight of the two wells** right, so the table below gives, for every rule, the mass of
the left well, P(x < 0), averaged over runs, next to the true value.

In [ ]:
xs = bl.GRID.numpy(); win = (xs > -3.5) & (xs < 3.5)
fig, axs = plt.subplots(1, len(betas), figsize=(5.5 * len(betas), 4.3), constrained_layout=True, squeeze=False)
for ax, b in zip(axs[0], betas):
    r = runs[b][RUN_SHOW]
    ax.plot(xs[win], bl.log_density(r['theta_star'])[0][win], color=INK, lw=3.2, label='true p*')
    ax.plot(xs[win], bl.log_density(r['theta_mle'])[0][win], color=INK2, ls='--', lw=1.5, label='max-likelihood fit')
    for rule, (lab, col, ls, mk) in SHOW.items():
        th = theta_end(r, rule)
        if th is not None:
            ax.plot(xs[win], bl.log_density(th)[0][win], color=col, ls=ls, lw=1.8, label=lab)
    lt = bl.log_density(r['theta_star'])[0][win].numpy()
    ax.set_ylim(lt.max() - 12, lt.max() + 1.5)
    ax.set_title(f'β = {b:.2f}, run {RUN_SHOW}', color=INK); ax.set_xlabel('x'); ax.set_ylabel('log p(x)')
h, l = axs[0, 0].get_legend_handles_labels()
legend_above(fig, h, l, ncol=4)
plt.show()

left = (bl.GRID < 0).double()
def p_left(th):
    lp = bl.log_density(th)[0]
    return float((lp.exp() * left).sum() * bl.DX) if torch.isfinite(lp).all() else np.nan
rows = []
for b in betas:
    row = {'beta': b, 'true': p_left(runs[b][0]['theta_star']),
           'max-likelihood fit': np.mean([p_left(r['theta_mle']) for r in runs[b]])}
    for rule, (lab, *_) in SHOW.items():
        row[lab] = np.nanmean([p_left(theta_end(r, rule)) if theta_end(r, rule) is not None else np.nan for r in runs[b]])
    rows.append(row)
print('mass of the left well P(x < 0), mean over runs')
pd.DataFrame(rows).set_index('beta')

## 6. Error at t = 1 as a function of λ

One thin line per run, for the two weightings of the time term (uniform: blue, Guth: aqua). The
symbols mark the λ each rule picked in that run. The grey dotted line is the raw θ_t (λ = 0). The
right end of a curve rising again means too much smoothing (bias); the left end, too little (noise).

In [ ]:
lams = runs[betas[0]][0]['lams']; lpos = [l for l in lams if l > 0]
SCHED_COL = {'uniform': '#2a78d6', 'guth': '#1baf7a'}
MARK = {'cv_eo': ('s', 'even/odd CV'), f'cv_block{BMAX}': ('D', f'block CV, B = {BMAX}'), 'oracle_end': ('v', 'oracle')}
fig, axs = plt.subplots(1, len(betas), figsize=(5.5 * len(betas), 4.3), constrained_layout=True, squeeze=False)
for ax, b in zip(axs[0], betas):
    for r in runs[b]:
        for sched, col in SCHED_COL.items():
            ax.plot(lpos, [r['kl_end_true'][f'{sched}|{l:g}'] for l in lpos], color=col, lw=0.9, alpha=0.6)
            for suf, (mk, _) in MARK.items():
                lam = r['rules'].get(f'{sched}_{suf}', {}).get('lam')
                if lam:
                    ax.plot(lam, r['kl_end_true'][f'{sched}|{lam:g}'], marker=mk, color=col, mfc=SURF, mew=1.8, ls='')
        ax.axhline(r['rules']['lam0']['end_true'], color=MUTED, ls=':', lw=0.8)
    ax.axhline(np.mean([r['kl_floor'] for r in runs[b]]), color=INK, ls='--', lw=1.2)
    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('λ'); ax.set_title(f'β = {b:.2f}', color=INK)
axs[0, 0].set_ylabel('D_KL(p_θ* ‖ p_Θ(λ)) at t = 1')
hs = [plt.Line2D([], [], color=c, lw=1.5) for c in SCHED_COL.values()] + \
     [plt.Line2D([], [], marker=mk, color=INK2, mfc=SURF, ls='') for mk, _ in MARK.values()] + \
     [plt.Line2D([], [], color=MUTED, ls=':'), plt.Line2D([], [], color=INK, ls='--')]
legend_above(fig, hs, ['uniform weights', 'Guth weights'] + [n for _, n in MARK.values()] +
             ['raw θ_t (λ = 0)', 'floor (max-likelihood fit)'], ncol=4)
plt.show()

## 7. Chosen λ against the best λ

For each practical rule, how many decades its λ is away from the oracle's λ (same weighting),
log₁₀(λ_chosen / λ_best), one dot per run. 0 means it found the best λ of the grid, negative means it
smoothed too little. A rule that picked λ = 0 has its own row at the bottom ("λ = 0"), and a run
whose best λ is 0 its own row at the top.

In [ ]:
lpos_ = [l for l in lams if l > 0]
span = int(np.ceil(np.log10(max(lpos_) / min(lpos_)))) if len(lpos_) > 1 else 1
Y0, Y1 = -span - 1, span + 1                                  # rows for "chose 0" and "best is 0"
prac = ['uniform_cv_eo'] + [f'uniform_cv_block{B}' for B in BLOCKS] + ['guth_cv_eo'] + [f'guth_cv_block{B}' for B in BLOCKS]
fig, axs = plt.subplots(1, len(betas), figsize=(5.5 * len(betas), 4.3), constrained_layout=True, squeeze=False, sharey=True)
for ax, b in zip(axs[0], betas):
    for j, rule in enumerate(prac):
        sched = rule.split('_')[0]
        d = []
        for r in runs[b]:
            lc, lb = r['rules'].get(rule, {}).get('lam'), r['rules'][f'{sched}_oracle_end']['lam']
            if lc is None:
                continue
            d.append(Y0 if lc == 0 else (Y1 if lb == 0 else np.log10(lc / lb)))
        d = np.array(d, dtype=float)
        ax.scatter(j + rng.uniform(-0.15, 0.15, len(d)), d,
                   color=SCHED_COL[sched], s=30, edgecolors=SURF, linewidths=0.8, zorder=3)
    ax.axhline(0, color=INK2, lw=0.8)
    for y in (Y0 + 0.5, Y1 - 0.5):
        ax.axhline(y, color=GRIDC, lw=2)
    ax.set_yticks([Y0] + list(range(-span, span + 1)) + [Y1])
    ax.set_yticklabels(['λ = 0'] + [str(v) for v in range(-span, span + 1)] + ['best = 0'])
    ax.set_xticks(range(len(prac)))
    ax.set_xticklabels([p.replace('uniform_', 'U ').replace('guth_', 'G ').replace('cv_', '') for p in prac], rotation=45)
    ax.set_title(f'β = {b:.2f}', color=INK)
axs[0, 0].set_ylabel('log₁₀(λ chosen / λ best)')
plt.show()

## 8. Is the noise of θ_t correlated in time?

The residual θ_t - (reference) along each time window: its autocorrelation at lags of 1, 10, 100
and 1000 steps (median over the four components and over runs), and τ_int, roughly *how many
consecutive steps share the same noise* (1 = independent steps, the case where even/odd CV is
safe). E[z²] compares the size of the noise with the noise model of `lam_select_cv` (1 = as
predicted). At σ = 10 the small difference between MGD's θ_t and the reference (0.13 % in the 1-D
check) is slow in t, so it also shows up here as correlation at long lags.

In [ ]:
rows = []
for b in betas:
    for w in runs[b][0]['noise']:
        g = lambda k: np.median([r['noise'][w][k] for r in runs[b] if w in r['noise']])
        rows.append({'beta': b, 'window': w, 'steps': int(g('steps')), 'acf 1': g('acf1'), 'acf 10': g('acf10'),
                     'acf 100': g('acf100'), 'acf 1000': g('acf1000'), 'τ_int (lags ≤ 100)': g('tau_int_100'),
                     'τ_int (all lags)': g('tau_int'), 'E[z²]': g('z2_median')})
noise = pd.DataFrame(rows).set_index(['beta', 'window'])
noise